# Deep Key Search in Nested JSON

**Company:** MongoDB (GothamLoop question bank) · **Category:** Coding · **Tags:** Live Screen · **Difficulty/Frequency:** Common (6/10)

> The source page for this question carries the **short format** — statement plus one guidance paragraph, no separate hints/answer/walkthrough. See [`README.md`](README.md); everything below is worked out here.

## Concepts

**What this problem is really testing:**
- Recognising a JSON document as a **tree**, and "search at any depth" as a **tree traversal**
- Writing clean **recursion** over a structure with more than one node type (dict, list, scalar)
- The **sentinel** pattern — how to say "not found" when *every* value, including `None`, is a legal answer

**First-principles primer — what is each piece?**

- **JSON as a tree.** An object `{...}` and an array `[...]` are *internal nodes* — they contain other nodes. Strings, numbers, booleans and `null` are *leaves*. There is no cycle in JSON parsed from text, so it is a tree, not a general graph.
- **DFS (depth-first search).** Follow one branch as deep as it goes, then back up and try the next. Recursion gives it to you for free: each call frame *is* one level of the stack. **Pre-order** DFS means you inspect a node *before* descending into its children — which is exactly "check the keys at this level, then go deeper".
- **Sentinel.** A unique object created only to mean "nothing here" — `_MISSING = object()`. Because it is compared with `is`, nothing in the data can ever be mistaken for it.
- **Depth vs. size.** The recursion stack grows with the **depth** `d` (how deeply nested), not the **size** `N` (how many nodes total). A document with a million flat keys costs one stack frame; one nested a million deep overflows.

**The insight the problem is built around:**

> `None` is a **valid JSON value**, so it cannot double as your "not found" answer.

`{"a": None}` is a perfectly good document. If `deep_search` returns `None` both for "the key exists and its value is null" and for "the key is not here", then:

- the **caller** cannot tell the two apart, and
- worse, the **recursion itself** cannot either — so after genuinely finding `{"a": None}` it concludes "not found" and keeps searching, eventually returning the wrong branch's value or `None` from the wrong place.

The same trap catches every falsy value. Writing `if found:` instead of `if found is not _MISSING:` breaks on `0`, `""`, `False` and `[]` — all legal JSON.

**Simple worked example.** Searching the sample document for `lead_strategist`:

```
{}                                 <- level 0: keys are id, company_name, company_details, active. No match.
  company_details                  <- level 1: keys are location, departments. No match.
    location                       <- level 2: street, city, state, zipcode. No match. Back up.
    departments                    <- level 2: engineering, marketing. No match.
      engineering                  <- level 3: team_count, lead_developer. No match. Back up.
      marketing                    <- level 3: team_count, LEAD_STRATEGIST  ->  "Sarah Chen"  ✅
```

Note that `departments` itself would have matched at level 2 — and the answer would be the **whole object**, both departments included. The return type is "any JSON value", not "a scalar".

## Problem Statement

Given a JSON document whose values may be strings, numbers, booleans, `null`, lists, or nested objects, and a query key, search the **entire structure at any depth** and return the value for that key.

**Required behaviour**

- `search("departments")` returns the **whole** `departments` object, both nested departments included.
- `search("lead_strategist")` returns `"Sarah Chen"`.

**The three questions to ask the interviewer** (the source page flags these explicitly):

1. The key appears **more than once** — return the first match, or all of them?
2. Must **lists** be searched?
3. What is the **return type** when the match is itself an object?

The answers taken here: **first match in pre-order traversal**, **yes, descend through lists**, and **return the live sub-object unchanged**. A `find_all` variant is built below for the first question.

In [ ]:
from typing import Any, Dict, Iterator, List, Optional, Tuple

DOC: Dict[str, Any] = {
    "id": 101,
    "company_name": "TechNova Solutions",
    "company_details": {
        "location": {
            "street": "500 Innovation Way",
            "city": "San Francisco",
            "state": "CA",
            "zipcode": "94105",
        },
        "departments": {
            "engineering": {"team_count": 5, "lead_developer": "Alex Rivera"},
            "marketing": {"team_count": 3, "lead_strategist": "Sarah Chen"},
        },
    },
    "active": True,
}

# A unique object that can never equal any JSON value. Compared with `is`, never `==`.
_MISSING = object()

### Approach 1 — Naive (flatten the whole document first, then look up)

**Idea:** walk the entire structure once, collecting every `key -> value` pair into a flat dictionary, then answer the query from that.

It is easy to reason about, and it is genuinely the right choice if you will run **many** queries against the **same** document. But as a one-shot answer it has two real problems:

- It **always** visits every node, even when the answer sits in the first key. No early exit.
- Flattening **collides** duplicate keys. `team_count` exists twice; a flat dict keeps only one, and which one depends on your traversal order. Information is destroyed before the query is even asked.

**Time complexity:** O(N) to flatten, then O(1) per query — but O(N) even for a query that a search would answer in one step.

**Space complexity:** **O(N)** — a second copy of every key in the document.

In [ ]:
def flatten_keys(obj: Any, out: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    """Collect every key anywhere in the document into one flat dict.

    Deliberately keeps the FIRST value seen for a repeated key, to match the
    pre-order semantics of the search below - but the collision is real either way.
    """
    if out is None:
        out = {}
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k not in out:                 # first-wins; a plain assignment would be last-wins
                out[k] = v
            flatten_keys(v, out)
    elif isinstance(obj, list):
        for item in obj:
            flatten_keys(item, out)
    return out


def deep_search_flatten(doc: Any, key: str) -> Any:
    return flatten_keys(doc).get(key, _MISSING)   # rebuilt on EVERY call

### Approach 2 — Optimal (recursive pre-order DFS with a sentinel)

**Idea:** check the keys at the current level first; only if none match, descend into each value in turn. Stop the instant something is found.

Three details carry the whole solution:

- **`key in obj` before recursing.** That is what makes it *pre-order*, and pre-order is what makes the result the **shallowest, earliest** match. Descend first and you get the deepest one instead — almost never what "find this key" means.
- **`found is not _MISSING`, never `if found:`.** The sentinel is compared by identity, so `0`, `False`, `""`, `[]` and `None` all pass through as genuine answers.
- **Lists are traversed but never match.** A list index is not a key. You descend through `[...]` looking for objects inside it, but a list itself can never be the thing the query names.

The returned object is the **live sub-object**, not a copy — which is precisely what "query `departments` returns the whole departments object" asks for.

**Time complexity:** **O(N)** worst case, but with **early exit** — a match near the front costs only the nodes visited to reach it.

**Space complexity:** **O(d)** — one stack frame per level of nesting, not per node.

In [ ]:
def deep_search(obj: Any, key: str) -> Any:
    """First match in pre-order DFS. Returns _MISSING if the key is nowhere."""
    if isinstance(obj, dict):
        if key in obj:                       # PRE-order: this level before any child
            return obj[key]
        for value in obj.values():
            found = deep_search(value, key)
            if found is not _MISSING:        # identity check - `if found:` would break on 0/""/None
                return found                 # early exit: stop the moment we have an answer
    elif isinstance(obj, list):
        for item in obj:                     # descend THROUGH lists; a list never matches
            found = deep_search(item, key)
            if found is not _MISSING:
                return found
    return _MISSING                          # scalars, and exhausted containers


def deep_get(obj: Any, key: str, default: Any = None) -> Any:
    """Caller-friendly wrapper: a real default instead of a private sentinel."""
    found = deep_search(obj, key)
    return default if found is _MISSING else found

### Approach 3 — Iterative DFS (no recursion limit)

**Idea:** the same traversal with an explicit stack instead of the call stack.

Why it matters: Python's default recursion limit is about 1000 frames. A document nested deeper than that — or one crafted by an attacker to be — kills the recursive version with `RecursionError`. The iterative version is bounded only by heap memory.

**The one subtlety:** a stack is LIFO, so pushing children left-to-right pops them right-to-left. Pushing them **reversed** restores the same visit order as the recursive version, so both implementations return the identical match. Getting this wrong is silent — the code still "works", it just picks a different duplicate.

**Time complexity:** O(N) worst case, with the same early exit.

**Space complexity:** O(N) worst case for the explicit stack — slightly worse than the recursion's O(d), which is the price of not being able to overflow.

In [ ]:
def deep_search_iterative(obj: Any, key: str) -> Any:
    stack: List[Any] = [obj]
    while stack:
        node = stack.pop()
        if isinstance(node, dict):
            if key in node:
                return node[key]
            # reversed(): a LIFO stack would otherwise visit children right-to-left
            stack.extend(reversed(list(node.values())))
        elif isinstance(node, list):
            stack.extend(reversed(node))
    return _MISSING

### Follow-up — every match, with its path

**Idea:** the first-match contract throws away information the caller often wants. `team_count` genuinely appears twice; returning `5` alone is a half-answer.

A generator that yields `(dotted_path, value)` for **every** occurrence fixes it:

- **A generator, not a list**, so a caller who only wants the first match still gets the early exit for free.
- **Paths make the result actionable** — `"company_details.departments.marketing.lead_strategist"` tells you *where*, which is what `jq`, JSONPath and MongoDB's own dotted field notation all return.
- **List elements are indexed** as `tags[0]`, keeping the path unambiguous.

Note it does **not** stop descending after a match: a key can legitimately appear inside the very sub-object it matched.

**Time complexity:** O(N) to enumerate everything (lazily).

**Space complexity:** O(d) for the recursion, plus whatever the caller keeps.

In [ ]:
def find_all(obj: Any, key: str, path: str = "") -> Iterator[Tuple[str, Any]]:
    """Yield (dotted path, value) for EVERY occurrence of `key`, in pre-order."""
    if isinstance(obj, dict):
        for k, v in obj.items():
            here = f"{path}.{k}" if path else k
            if k == key:
                yield here, v
            yield from find_all(v, key, here)      # keep going: nested repeats count too
    elif isinstance(obj, list):
        for i, item in enumerate(obj):
            yield from find_all(item, key, f"{path}[{i}]")


def get_by_path(obj: Any, dotted: str) -> Any:
    """Exact descent along a known path: O(depth), no searching at all."""
    node = obj
    for part in dotted.split("."):
        while "[" in part:                          # handle tags[0][1]
            name, _, rest = part.partition("[")
            if name:
                node = node[name]
            idx, _, part = rest.partition("]")
            node = node[int(idx)]
            part = part.lstrip(".")
        if part:
            node = node[part]
    return node

## Verification

Run the two required behaviours from the statement, then the cases that actually break implementations: `null` values, falsy values, duplicate keys, lists, and deep nesting.

In [ ]:
SEARCHERS = [deep_search, deep_search_iterative, deep_search_flatten]

# --- The two required behaviours ---
departments = deep_search(DOC, "departments")
assert departments == {
    "engineering": {"team_count": 5, "lead_developer": "Alex Rivera"},
    "marketing": {"team_count": 3, "lead_strategist": "Sarah Chen"},
}, "querying a key whose value is an object must return the WHOLE object"
assert departments is DOC["company_details"]["departments"], "return the live sub-object, not a copy"
assert deep_search(DOC, "lead_strategist") == "Sarah Chen"

# All three implementations agree on the sample document
for fn in SEARCHERS:
    assert fn(DOC, "lead_strategist") == "Sarah Chen", fn.__name__
    assert fn(DOC, "city") == "San Francisco", fn.__name__
    assert fn(DOC, "id") == 101, fn.__name__                 # top level
    assert fn(DOC, "active") is True, fn.__name__            # a boolean, not a truthiness test
    assert fn(DOC, "nope") is _MISSING, fn.__name__          # absent

# --- THE bug: None is a legal value, so it cannot mean "not found" ---
null_doc = {"a": {"b": None}}
for fn in SEARCHERS:
    assert fn(null_doc, "b") is None, fn.__name__            # found, and its value IS None
    assert fn(null_doc, "zzz") is _MISSING, fn.__name__      # genuinely absent - a DIFFERENT answer
assert deep_get(null_doc, "b", default="fallback") is None, "a real null must not become the default"
assert deep_get(null_doc, "zzz", default="fallback") == "fallback"

# --- Falsy values must survive: `if found:` would drop every one of these ---
falsy = {"outer": {"zero": 0, "empty_str": "", "false": False, "empty_list": [], "empty_dict": {}}}
for fn in SEARCHERS:
    assert fn(falsy, "zero") == 0, fn.__name__
    assert fn(falsy, "empty_str") == "", fn.__name__
    assert fn(falsy, "false") is False, fn.__name__
    assert fn(falsy, "empty_list") == [], fn.__name__
    assert fn(falsy, "empty_dict") == {}, fn.__name__

# --- Lists are traversed; list indices are not keys ---
listy = {"tags": [{"name": "alpha"}, {"name": "beta"}], "nested": [[{"deep": 1}]]}
for fn in (deep_search, deep_search_iterative):
    assert fn(listy, "name") == "alpha", (fn.__name__, "first match in order")
    assert fn(listy, "deep") == 1, (fn.__name__, "objects inside nested lists are reachable")
    assert fn(listy, "0") is _MISSING, (fn.__name__, "a list index is not a key")

# --- Duplicate keys: pre-order returns the FIRST, and both DFS versions agree on which ---
assert deep_search(DOC, "team_count") == 5, "engineering comes before marketing"
assert deep_search_iterative(DOC, "team_count") == 5, "the reversed() push must preserve order"

# --- Shallowest wins: pre-order checks the current level before descending ---
shadow = {"x": "shallow", "inner": {"x": "deep"}}
for fn in SEARCHERS:
    assert fn(shadow, "x") == "shallow", fn.__name__

# --- Edge cases ---
for fn in SEARCHERS:
    assert fn({}, "a") is _MISSING, fn.__name__              # empty document
    assert fn([], "a") is _MISSING, fn.__name__              # bare empty list
    assert fn("just a string", "a") is _MISSING, fn.__name__ # a scalar document
    assert fn(None, "a") is _MISSING, fn.__name__            # None document
    assert fn(42, "a") is _MISSING, fn.__name__

# --- Deep nesting: the iterative version survives where recursion would not ---
deep_doc: Any = {"leaf": "bottom"}
for _ in range(5000):
    deep_doc = {"down": deep_doc}
assert deep_search_iterative(deep_doc, "leaf") == "bottom", "no recursion limit"
try:
    deep_search(deep_doc, "leaf")
except RecursionError:
    pass          # expected: this is exactly why the iterative version exists
else:
    pass          # a raised recursion limit is fine too - the point is only that one CANNOT fail

# --- find_all: every occurrence, with paths ---
all_counts = list(find_all(DOC, "team_count"))
assert all_counts == [
    ("company_details.departments.engineering.team_count", 5),
    ("company_details.departments.marketing.team_count", 3),
], all_counts
assert list(find_all(DOC, "lead_strategist")) == [
    ("company_details.departments.marketing.lead_strategist", "Sarah Chen")
]
assert list(find_all(DOC, "absent")) == []
assert [p for p, _ in find_all(listy, "name")] == ["tags[0].name", "tags[1].name"]

# find_all's first result must match deep_search - the two contracts stay consistent
for k in ("team_count", "lead_strategist", "city", "departments"):
    assert next(iter(find_all(DOC, k)))[1] == deep_search(DOC, k), k

# --- Every path find_all reports must actually resolve back to its value ---
for k in ("team_count", "lead_strategist", "city", "zipcode", "departments"):
    for path, value in find_all(DOC, k):
        assert get_by_path(DOC, path) is value or get_by_path(DOC, path) == value, path
assert get_by_path(DOC, "company_details.location.city") == "San Francisco"
assert get_by_path(listy, "tags[1].name") == "beta"

print("All assertions passed.")

## Discussion — remaining follow-up directions

- **BFS instead of DFS.** Pre-order DFS returns the first match *in key order*, which is not the same as the **shallowest** match. Replace the stack with a `collections.deque` and pop from the left, and you get level-by-level search — usually the better default for "find this config value", because a key nearer the root is more likely to be the one meant. Same O(N); the space becomes O(width) instead of O(depth).
- **Dotted-path lookup vs. search.** `get_by_path` above answers `company_details.location.city` in **O(d)** — it walks straight down, never searching. When the caller knows where the value lives, this is strictly better, and it is exactly MongoDB's own dotted field notation. Search is for when you *don't* know.
- **Many queries against one document.** Then Approach 1's instinct is right, but do it properly: one O(N) pass building `key -> [(path, value), ...]` (a list, so duplicates survive), after which every query is O(1). That is the same "invert the mapping" move as the [Inverted Index](../1.%20Inverted_Index/1.%20Inverted_Index.ipynb) problem — and the reason a database builds an index instead of scanning.
- **Cycles.** JSON parsed from text cannot contain a cycle, but a hand-built Python `dict` can (`d["self"] = d`), and both DFS versions would spin forever. Guard with a `visited` set of `id(node)` if the input is not guaranteed to come from a parser. Say which assumption you are making rather than silently relying on it.
- **Documents too large for memory.** Switch to a streaming, event-driven parser (`ijson`, or a SAX-style reader) that fires a callback per key as it reads. You never hold the tree, memory is O(depth), and you can stop reading the file the moment you match.
- **Matching on more than a key name.** "Every `lead_*` key", or "every value greater than 4" — the traversal is unchanged; only the predicate at each node moves from `k == key` to `pred(k, v)`. Factoring the walk apart from the test is what turns this one function into a small query engine.

## Empirical complexity check

Compare **flatten-then-lookup** (Approach 1, rebuilds a full O(N) dictionary on every call) against **DFS with early exit** (Approach 2) — running a fixed number of queries against a document that doubles in size.

The queries deliberately target keys near the **front** of the document, which is where the early exit pays: DFS stops as soon as it matches, while flattening always visits every node first.

| Growth when the document doubles | What it means |
|---|---|
| ~2x | linear — the whole document is visited on every query |
| ~1x | the early exit dominates — the work is bounded by the depth to the match, not the size |

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

QUERIES = 30


def make_document(n):
    """A document with n leaf keys, spread across n/10 sibling sub-objects."""
    doc = {}
    for i in range(max(1, n // 10)):
        doc[f"section_{i}"] = {f"field_{i}_{j}": j for j in range(10)}
    return (doc,)


def run_flatten(doc):
    for q in range(QUERIES):
        deep_search_flatten(doc, f"field_0_{q % 10}")     # rebuilds the flat dict EVERY call


def run_dfs(doc):
    for q in range(QUERIES):
        deep_search(doc, f"field_0_{q % 10}")             # early exit: match is in the first section


benchmark(
    {"Approach 1 - flatten then look up": run_flatten,
     "Approach 2 - DFS with early exit": run_dfs},
    make_document,
    sizes=[1000, 2000, 4000, 8000],
    repeats=2,
)

## Patterns learned

- **Nested containers are trees; "at any depth" means traversal.** Once you name it that way, the shape of the code is decided — the only remaining choices are pre-order vs. post-order and DFS vs. BFS.
- **Pre-order gives you the shallowest match.** Inspect the node before descending. Post-order returns the deepest match instead — a real behavioural difference, not a style preference.
- **Never overload a valid value as your error signal.** `None` is legal JSON, so "not found" needs a sentinel, a `(found, value)` pair, or an exception. This bug is invisible in testing until the day a document contains a `null`.
- **Identity checks (`is`) for sentinels, never truthiness.** `if found:` silently discards `0`, `""`, `False` and `[]`.
- **Recursion depth is the input's depth, not its size.** Cheap for wide documents, fatal for deep ones. Keep the iterative version in your pocket for the "what if it is 10,000 levels deep?" follow-up.
- **When a stack replaces recursion, push children reversed.** LIFO flips the order; reversing flips it back. Silent bug otherwise.
- **Return paths, not just values.** A path is actionable — it tells the caller *where*, survives duplicates, and turns a lookup into something you can update through.
- **One query: search. Many queries: index.** The break-even is immediate — the moment you will ask more than a couple of questions of the same document, build the map once.